In [1]:
from json import load
import autogen
from dotenv import load_dotenv
import os
from openai import OpenAI
load_dotenv()


True

In [2]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
class DignosticAgent(autogen.AssistantAgent):
    def __init__(self, name: str, model: str= 'gpt-4o-mini'):
        super().__init__(name=name)
        self.model = model
     
    def analyse_issue(self, issue):
        """Analyzes the issue and returns a diagnosis."""

        if any(keyword in issue.lower() for keyword in ["password", "slow internet", "software update"]):
            return "simple"
        elif any(keyword in issue.lower() for keyword in ["network failure", "hardware crash", "security breach"]):
            return "critical"
        else:
            return "complex"

    

In [4]:
class ResolutionAgent(autogen.AssistantAgent):
    def __init__(self, name= "ResolutionAgent", model='gpt-4o-mini'):
        super().__init__(name=name)
        self.model = model

    def resolve_issue(self, issue):
        """Uses the GPT to troubleshoot complex problems. (if you have custom solution RAG please integrate it here)"""
        response = client.chat.completions.create(
            model=self.model,
            messages=[{"role": "system", "content":"You are a helpful assistant that can troubleshoot complex problems."},
            {"role": "user", "content": f"User's issue: {issue}\n Resolution:"}]
        )
        return response.choices[0].message.content.strip()

In [5]:
class EscalationAgent(autogen.AssistantAgent):
    def __init__(self, name= "EscalationAgent"):
        super().__init__(name=name)

    def esclate_issue(self, issue):
        return f"The issue '{issue}' is critical. Escalating to the IT team."


In [6]:
diagonastic_agent = DignosticAgent(name="DiagonasticAgent")
resolution_agent = ResolutionAgent(name="ResolutionAgent")
escalation_agent = EscalationAgent(name="EscalationAgent")





In [7]:
user_issue = " My laptop hardware crash"
issue_type = diagonastic_agent.analyse_issue(user_issue)

In [8]:
issue_type

'critical'

In [9]:
if issue_type in ["simple", 'complex']:
    response = resolution_agent.resolve_issue(user_issue)
else:
    response = escalation_agent.esclate_issue(user_issue)

In [10]:
response

"The issue ' My laptop hardware crash' is critical. Escalating to the IT team."